# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ANEMBHARGAV/flyrank_ml_intern_w1/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

My baseline rule
My baseline prioritizes pages for manual content-refresh review using two observable signals: staleness and search-performance efficiency.

First, older pages receive a higher staleness score as a prioritization heuristic for freshness review. The signal check was mixed, so staleness is treated as a directional heuristic rather than a confirmed relationship. Second, pages with relatively low CTR for their observed average position receive a higher search-efficiency score.

The final baseline score combines these two signals. Pages are ranked from highest to lowest score.

The rule produces one reason code:

STALE_AND_LOW_CTR — the page is both old and has relatively low CTR for its position.
STALE — staleness is the main signal.
LOW_CTR — CTR relative to position is the main signal.
REVIEW — neither signal is dominant, but the page still enters the ranked queue.
Action labels are `REVIEW_CONTENT` when the baseline score is at least 0.5 and `MONITOR` otherwise.

The score uses only information available at the scoring point. No future-window outcome or label-derived

In [2]:
# Load the starter dataset from the GitHub repository

import os
import pandas as pd
import numpy as np
import subprocess

REPO_URL = "https://github.com/ANEMBHARGAV/flyrank_ml_intern_w1"
REPO_DIR = "/content/flyrank_ml_intern_w1"

# Clone the repository if it is not already available
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)

# Load the dataset
df = pd.read_csv(
    os.path.join(REPO_DIR, "data/raw/content_refresh_anonymized.csv")
)

print("Rows:", len(df))
print("Columns:")
print(df.columns.tolist())

Rows: 30000
Columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [3]:
# Inspect the two baseline signals and their available categories

print("Freshness tiers:")
print(df["freshness_tier"].value_counts(dropna=False).sort_index())

print("\nPosition tiers:")
print(df["position_tier"].value_counts(dropna=False).sort_index())

print("\nStaleness range:")
print(df["days_since_last_update"].describe())

print("\nCTR range:")
print(df["ctr"].describe())

print("\nAverage position range:")
print(df["avg_position"].describe())

Freshness tiers:
freshness_tier
0-30      20480
181+        174
31-90       175
91-180     9171
Name: count, dtype: int64

Position tiers:
position_tier
deep         1319
page_1      11814
page_3_5     7242
striking     7304
top_3        2321
Name: count, dtype: int64

Staleness range:
count    30000.000000
mean        46.098300
std         42.078709
min          1.000000
25%         20.000000
50%         20.000000
75%        104.000000
max        373.000000
Name: days_since_last_update, dtype: float64

CTR range:
count    30000.000000
mean         0.510733
std          3.279162
min          0.000000
25%          0.000000
50%          0.070000
75%          0.290000
max        100.000000
Name: ctr, dtype: float64

Average position range:
count    30000.00000
mean        16.34238
std         15.21679
min          0.00000
25%          6.20000
50%         10.80000
75%         22.30000
max        245.00000
Name: avg_position, dtype: float64


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [7]:
# Signal check against the observed trend flag.
# trend_direction is used only for signal validation, not for scoring.

# Signal 1: staleness
df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[0, 30, 90, 180, np.inf],
    labels=["0-30", "31-90", "91-180", "181+"],
    include_lowest=True
)

staleness_check = (
    df.groupby("staleness_bucket", observed=False)
      .agg(
          n=("content_id", "size"),
          down_rate=("trend_direction", lambda x: (x == "down").mean())
      )
      .reset_index()
)

print("Signal 1 — Staleness vs observed trend flag")
print(staleness_check.to_string(index=False))


# Signal 2: CTR relative to position
position_median_ctr = (
    df.groupby("position_tier")["ctr"]
      .transform("median")
)

df["ctr_position_bucket"] = np.where(
    df["ctr"] < position_median_ctr,
    "Below position median CTR",
    "At/above position median CTR"
)

ctr_check = (
    df.groupby("ctr_position_bucket")
      .agg(
          n=("content_id", "size"),
          down_rate=("trend_direction", lambda x: (x == "down").mean())
      )
      .reset_index()
)

print("\nSignal 2 — CTR vs position")
print(ctr_check.to_string(index=False))

Signal 1 — Staleness vs observed trend flag
staleness_bucket     n  down_rate
            0-30 20480   0.511377
           31-90   175   0.588571
          91-180  9171   0.611057
            181+   174   0.471264

Signal 2 — CTR vs position
         ctr_position_bucket     n  down_rate
At/above position median CTR 16921   0.502630
   Below position median CTR 13079   0.593088


### Signal verdicts

**Signal 1 — Staleness: MIXED**

The observed down rate increases from 51.1% for pages updated within 30 days to 61.1% for pages in the 91–180 day bucket, but it falls to 47.1% for the 181+ bucket. This does not show a consistent monotonic relationship, so I treat staleness as a mixed signal.

**Signal 2 — CTR vs position: CONFIRMED**

Pages with CTR below the median CTR for their position tier have a higher observed down rate (59.3%) than pages at or above the position median (50.3%). This supports using CTR relative to position as a baseline signal.

The observed trend flag is used only to check the signals. It is not used as an input to the baseline score.

In [9]:
# Build the baseline ranked queue

# Signal 1: staleness score (0 to 1)
df["stale_score"] = (
    df["days_since_last_update"] / df["days_since_last_update"].max()
).clip(0, 1)

# Signal 2: low CTR relative to average position
position_ctr_median = (
    df.groupby("position_tier")["ctr"]
      .transform("median")
)

df["low_ctr_score"] = (
    (position_ctr_median - df["ctr"]) /
    position_ctr_median.replace(0, np.nan)
).clip(lower=0, upper=1).fillna(0)

# Final baseline score
df["baseline_score"] = (
    0.5 * df["stale_score"] +
    0.5 * df["low_ctr_score"]
)

# One reason code
df["reason_code"] = np.select(
    [
        (df["stale_score"] >= 0.5) & (df["low_ctr_score"] >= 0.5),
        df["stale_score"] >= 0.5,
        df["low_ctr_score"] >= 0.5
    ],
    [
        "STALE_AND_LOW_CTR",
        "STALE",
        "LOW_CTR"
    ],
    default="REVIEW"
)

# Action label
df["action"] = np.where(
    df["baseline_score"] >= 0.5,
    "REVIEW_CONTENT",
    "MONITOR"
)

# Rank highest priority first
df = df.sort_values(
    ["baseline_score", "impressions_90d"],
    ascending=[False, False]
).reset_index(drop=True)

df["rank"] = np.arange(1, len(df) + 1)

# Select output columns
output = df[
    [
        "rank",
        "content_id",
        "client_id",
        "baseline_score",
        "reason_code",
        "action",
        "days_since_last_update",
        "ctr",
        "avg_position",
        "impressions_90d"
    ]
]

# Write the required CSV
os.makedirs(os.path.join(REPO_DIR, "work/outputs"), exist_ok=True)

output.to_csv(
    os.path.join(REPO_DIR, "work/outputs/baseline_action_score.csv"),
    index=False
)

print("Ranked queue created.")
print("Rows:", len(output))
print("Top 10:")
print(output.head(10).to_string(index=False))
print("\nSaved to:")
print("work/outputs/baseline_action_score.csv")

Ranked queue created.
Rows: 30000
Top 10:
 rank           content_id         client_id  baseline_score       reason_code         action  days_since_last_update  ctr  avg_position  impressions_90d
    1 content_55a5b1c46474 client_4ec9599fc2        1.000000 STALE_AND_LOW_CTR REVIEW_CONTENT                     373  0.0           7.5               35
    2 content_f6fdf87348f6 client_4ec9599fc2        1.000000 STALE_AND_LOW_CTR REVIEW_CONTENT                     373  0.0          32.5                2
    3 content_1b4ec72dafd4 client_4ec9599fc2        0.998660 STALE_AND_LOW_CTR REVIEW_CONTENT                     372  0.0           7.0                2
    4 content_8d56efff1e71 client_4ec9599fc2        0.998660 STALE_AND_LOW_CTR REVIEW_CONTENT                     372  0.0          35.0                1
    5 content_e2b702f4f92b client_4ec9599fc2        0.947721 STALE_AND_LOW_CTR REVIEW_CONTENT                     334  0.0           9.3               30
    6 content_06e19c6486b0 client_

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [5]:
# Review the top 20 ranked pages

top20 = output.head(20).copy()

def confidence_note(row):
    if row["reason_code"] == "STALE_AND_LOW_CTR":
        return "Both baseline signals are present"
    elif row["reason_code"] == "STALE":
        return "Staleness is the main signal"
    elif row["reason_code"] == "LOW_CTR":
        return "Low CTR is the main signal"
    else:
        return "Weak baseline signal"

def what_would_make_wrong(row):
    if row["reason_code"] == "STALE_AND_LOW_CTR":
        return "Wrong if the low CTR is expected for its position or the page is intentionally unchanged"
    elif row["reason_code"] == "STALE":
        return "Wrong if the page is old but still performs well and does not need a refresh"
    elif row["reason_code"] == "LOW_CTR":
        return "Wrong if the CTR is normal for this position or query intent"
    else:
        return "Wrong if neither signal indicates a useful refresh opportunity"

top20["confidence_note"] = top20.apply(confidence_note, axis=1)
top20["what_would_make_wrong"] = top20.apply(what_would_make_wrong, axis=1)

review = top20[
    [
        "rank",
        "content_id",
        "action",
        "reason_code",
        "confidence_note",
        "what_would_make_wrong"
    ]
]

print("Top-20 Review")
print(review.to_string(index=False))

Top-20 Review
 rank           content_id         action       reason_code                   confidence_note                                                                    what_would_make_wrong
    1 content_55a5b1c46474 REVIEW_CONTENT STALE_AND_LOW_CTR Both baseline signals are present Wrong if the low CTR is expected for its position or the page is intentionally unchanged
    2 content_f6fdf87348f6 REVIEW_CONTENT STALE_AND_LOW_CTR Both baseline signals are present Wrong if the low CTR is expected for its position or the page is intentionally unchanged
    3 content_1b4ec72dafd4 REVIEW_CONTENT STALE_AND_LOW_CTR Both baseline signals are present Wrong if the low CTR is expected for its position or the page is intentionally unchanged
    4 content_8d56efff1e71 REVIEW_CONTENT STALE_AND_LOW_CTR Both baseline signals are present Wrong if the low CTR is expected for its position or the page is intentionally unchanged
    5 content_e2b702f4f92b REVIEW_CONTENT STALE_AND_LOW_CTR Both baseli

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### Weak picks

Some top-ranked pages have very low impression volume, including pages with only 1–5 impressions over 90 days. These may be weak review candidates because there is little search activity to support a meaningful refresh decision.

They are still ranked highly by the baseline because they are very stale and have zero CTR. This shows a limitation of the baseline: it does not include a minimum-volume rule.

In [8]:
# Weak picks + leakage check

# Identify potentially weak top-20 picks:
# pages with very low impressions may be less useful for immediate review
weak_picks = top20[
    top20["impressions_90d"] <= top20["impressions_90d"].quantile(0.25)
].copy()

print("Potentially weak picks from the top 20:")
print(
    weak_picks[
        [
            "rank",
            "content_id",
            "baseline_score",
            "reason_code",
            "action",
            "impressions_90d",
            "days_since_last_update",
            "ctr",
            "avg_position"
        ]
    ].to_string(index=False)
)

# Leakage check
forbidden_columns = [
    "trend_direction",
    "trend_pct",
    "is_declining_label"
]

used_for_score = [
    "days_since_last_update",
    "ctr",
    "avg_position"
]

leaked_used = [
    col for col in forbidden_columns
    if col in used_for_score
]

print("\nLeakage check:")
print("Score features:", used_for_score)
print("Forbidden outcome fields used in score:", leaked_used)
print("Leakage check passed:", len(leaked_used) == 0)

Potentially weak picks from the top 20:
 rank           content_id  baseline_score       reason_code         action  impressions_90d  days_since_last_update  ctr  avg_position
    2 content_f6fdf87348f6        1.000000 STALE_AND_LOW_CTR REVIEW_CONTENT                2                     373  0.0          32.5
    3 content_1b4ec72dafd4        0.998660 STALE_AND_LOW_CTR REVIEW_CONTENT                2                     372  0.0           7.0
    4 content_8d56efff1e71        0.998660 STALE_AND_LOW_CTR REVIEW_CONTENT                1                     372  0.0          35.0
   16 content_ab18b5811c02        0.908847 STALE_AND_LOW_CTR REVIEW_CONTENT                5                     305  0.0          29.8
   20 content_84d12054c0c0        0.907507 STALE_AND_LOW_CTR REVIEW_CONTENT                1                     304  0.0           8.0

Leakage check:
Score features: ['days_since_last_update', 'ctr', 'avg_position']
Forbidden outcome fields used in score: []
Leakage check passe

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.